In [ ]:
import scanpy as sc
import liana as li
import pandas as pd
import numpy as np

# ── Load raw counts ───────────────────────────────────────────────────
adata = sc.read_h5ad(
    '/path/to/data/'
    'Analyses_Erickson_Junyi/Thesis_Raw_Combined_Master.h5ad'
)

# ── Load niche labels ─────────────────────────────────────────────────
adata_mean = sc.read_h5ad(
    '/path/to/data/'
    'Cell2location/spatial_mapping/exploration_mean/'
    'spatial_compositional_space_mean.h5ad'
)

# ── Subset to QC-passed spots and transfer niche labels ───────────────
adata = adata[adata_mean.obs_names].copy()
adata.obs['niche'] = adata_mean.obs['joint_alpha_0.9'].astype(str)

print(adata)
print('\nNiche distribution:')
print(adata.obs['niche'].value_counts().sort_index())

# ── Check what LIANA needs ────────────────────────────────────────────
print('\nLIANA available methods:')
print(li.mt.show_methods())

print('\nLIANA available resources:')
print(li.resource.show_resources())

In [ ]:
import liana as li

# ── Normalise ─────────────────────────────────────────────────────────
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# ── Exclude technical niches 0 and 3 ─────────────────────────────────
biological_niches = ['1', '2', '4', '5', '6']
adata_bio = adata[adata.obs['niche'].isin(biological_niches)].copy()
print(f"Spots after excluding technical niches: {adata_bio.n_obs}")

# ── Run LIANA ─────────────────────────────────────────────────────────
li.mt.rank_aggregate(
    adata_bio,
    groupby='niche',
    resource_name='consensus',
    expr_prop=0.1,       # ligand or receptor must be expressed in >10% of spots
    min_cells=50,        # minimum spots per niche
    verbose=True,
    use_raw=False
)

print('LIANA complete')
print(adata_bio.uns['liana_res'].head(10))

In [ ]:
import liana as li
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Get results ───────────────────────────────────────────────────────
liana_res = adata_bio.uns['liana_res']
print(f"Total interactions: {len(liana_res)}")
print(f"Unique ligand-receptor pairs: {liana_res['ligand_complex'].nunique()}")

# Save full results
liana_res.to_csv('LIANA_results_all_niches.csv', index=False)
print('Saved full results')

# ── Filter significant interactions ───────────────────────────────────
# magnitude_rank < 0.05 is the standard significance threshold
liana_sig = liana_res[liana_res['magnitude_rank'] < 0.05].copy()
print(f"\nSignificant interactions (magnitude_rank < 0.05): {len(liana_sig)}")

# ── Heatmap: number of significant interactions per niche pair ─────────
liana_sig['pair'] = liana_sig['source'].astype(str) + '_' + liana_sig['target'].astype(str)
interaction_counts = liana_sig.groupby(['source', 'target']).size().reset_index(name='n_interactions')

# Pivot to matrix
pivot = interaction_counts.pivot(index='source', columns='target', values='n_interactions').fillna(0)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    pivot,
    cmap='YlOrRd',
    annot=True, fmt='.0f',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Number of significant interactions'}
)
ax.set_title(
    'Cell-cell interactions per spatial niche pair\n(LIANA Rank Aggregate, magnitude_rank < 0.05)',
    fontsize=11, fontweight='bold'
)
ax.set_xlabel('Target niche')
ax.set_ylabel('Source niche')
plt.tight_layout()
plt.savefig('LIANA_interaction_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved interaction heatmap')

# ── Dotplot: top interactions ─────────────────────────────────────────
li.pl.dotplot(
    adata=adata_bio,
    colour='lr_means',
    size='specificity_rank',
    inverse_size=True,
    source_labels=['2', '5'],     # focus on tumour niches
    target_labels=['1', '4', '6'], # vs normal niches
    top_n=20,
    orderby='magnitude_rank',
    orderby_ascending=True,
    figure_size=(14, 8)
)
plt.savefig('LIANA_dotplot_tumour_vs_normal.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved dotplot')

fix dotplot: 

In [ ]:
# ── Top interactions from niche 5 (most active) ───────────────────────
niche5_interactions = liana_sig[
    (liana_sig['source'] == '5') | (liana_sig['target'] == '5')
].copy()

# Get top 20 by magnitude rank
top20 = niche5_interactions.nsmallest(20, 'magnitude_rank')
top20['interaction'] = top20['ligand_complex'] + ' → ' + top20['receptor_complex']
top20['niche_pair'] = 'Niche ' + top20['source'].astype(str) + ' → Niche ' + top20['target'].astype(str)

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    top20['niche_pair'],
    top20['interaction'],
    s=top20['lr_means'] * 50,
    c=top20['specificity_rank'],
    cmap='RdBu_r',
    vmin=0, vmax=0.05
)
plt.colorbar(scatter, ax=ax, label='Specificity rank (lower = more specific)')
ax.set_xlabel('Niche pair (source → target)', fontsize=11)
ax.set_ylabel('Ligand → Receptor', fontsize=11)
ax.set_title(
    'Top 20 ligand-receptor interactions involving niche 5\n(size = mean expression, colour = specificity)',
    fontsize=11, fontweight='bold'
)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('LIANA_top20_niche5.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved niche 5 top interactions')

# ── Top interactions from niche 2 (aggressive tumour) ────────────────
niche2_interactions = liana_sig[
    (liana_sig['source'] == '2') | (liana_sig['target'] == '2')
].copy()

top20_n2 = niche2_interactions.nsmallest(20, 'magnitude_rank')
top20_n2['interaction'] = top20_n2['ligand_complex'] + ' → ' + top20_n2['receptor_complex']
top20_n2['niche_pair'] = 'Niche ' + top20_n2['source'].astype(str) + ' → Niche ' + top20_n2['target'].astype(str)

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    top20_n2['niche_pair'],
    top20_n2['interaction'],
    s=top20_n2['lr_means'] * 50,
    c=top20_n2['specificity_rank'],
    cmap='RdBu_r',
    vmin=0, vmax=0.05
)
plt.colorbar(scatter, ax=ax, label='Specificity rank (lower = more specific)')
ax.set_xlabel('Niche pair (source → target)', fontsize=11)
ax.set_ylabel('Ligand → Receptor', fontsize=11)
ax.set_title(
    'Top 20 ligand-receptor interactions involving niche 2\n(size = mean expression, colour = specificity)',
    fontsize=11, fontweight='bold'
)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('LIANA_top20_niche2.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved niche 2 top interactions')

make one more plot showing interactions unique to each niche compared to all others. This highlights niche-specific communication rather than shared housekeeping interactions

In [ ]:
# ── Find interactions unique to each niche as source ─────────────────
for niche in ['1', '2', '4', '5', '6']:
    # Interactions where this niche is source
    niche_as_source = liana_sig[liana_sig['source'] == niche]
    
    # Interactions where NO other niche shows same ligand-receptor pair
    other_pairs = liana_sig[liana_sig['source'] != niche][['ligand_complex', 'receptor_complex']].apply(tuple, axis=1)
    niche_pairs = niche_as_source[['ligand_complex', 'receptor_complex']].apply(tuple, axis=1)
    
    unique_mask = ~niche_pairs.isin(other_pairs)
    unique_interactions = niche_as_source[unique_mask]
    
    print(f"Niche {niche} — unique interactions as source: {len(unique_interactions)}")
    if len(unique_interactions) > 0:
        print(unique_interactions[['ligand_complex', 'receptor_complex', 'target', 'magnitude_rank']].head(5).to_string())
    print()

magnitude_rank — the consensus significance score from LIANA, ranging from 0 to 1. Lower is more significant — closer to 0 means multiple methods agree this interaction is strongly active. It is essentially a combined p-value rank across all the methods LIANA runs (CellPhoneDB, NATMI, SingleCellSignalR etc.). All values shown are below 0.05 which is your significance threshold

Niche 2 has only 9 unique interactions but two of them (NECTIN4→TIGIT and HLA-F→LILRB2) are immune checkpoint/evasion signals — suggesting the aggressive tumour niche achieves immune evasion through a small set of highly specific inhibitory signals rather than broad communication

Niche 4 (Stroma): Your top differentially expressed gene (DEG) was MYL9. LIANA confirms that MYL9 isn't just sitting there; it is actively binding to CD69 on target cells.

Niche 1 (Periurethral): Your top DEG was LTF (Lactoferrin). LIANA confirms it is actively signaling to LRP1

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

# ── Recompute unique interactions per niche ───────────────────────────
niche_labels = {
    '1': 'Niche 1\nPeriurethral ductal',
    '2': 'Niche 2\nAggressive tumour',
    '4': 'Niche 4\nFibromuscular stroma',
    '5': 'Niche 5\nIncidental cancer',
    '6': 'Niche 6\nLuminal epithelium'
}

unique_per_niche = {}
for niche in ['1', '2', '4', '5', '6']:
    niche_as_source = liana_sig[liana_sig['source'] == niche]
    other_pairs = liana_sig[liana_sig['source'] != niche][
        ['ligand_complex', 'receptor_complex']
    ].apply(tuple, axis=1)
    niche_pairs = niche_as_source[
        ['ligand_complex', 'receptor_complex']
    ].apply(tuple, axis=1)
    unique_mask = ~niche_pairs.isin(other_pairs)
    unique_per_niche[niche] = niche_as_source[unique_mask]

# ── Top 5 unique interactions per niche ───────────────────────────────
top_unique = {}
for niche in ['1', '2', '4', '5', '6']:
    df = unique_per_niche[niche].nsmallest(5, 'magnitude_rank').copy()
    df['interaction'] = df['ligand_complex'] + ' → ' + df['receptor_complex']
    df['target_label'] = 'to Niche ' + df['target'].astype(str)
    top_unique[niche] = df

# ── Colours per niche ─────────────────────────────────────────────────
niche_colors = {
    '1': '#4e9af1',
    '2': '#d62728',
    '4': '#2ca02c',
    '5': '#ff7f0e',
    '6': '#9467bd'
}

# ── Figure: two panels ────────────────────────────────────────────────
# Panel A: bar chart of unique interaction counts
# Panel B: table of top 5 unique interactions per niche

fig = plt.figure(figsize=(18, 12))

# ── Panel A: bar chart ────────────────────────────────────────────────
ax1 = fig.add_subplot(1, 2, 1)

niches = ['1', '2', '4', '5', '6']
counts = [len(unique_per_niche[n]) for n in niches]
colors = [niche_colors[n] for n in niches]
labels = [niche_labels[n] for n in niches]

bars = ax1.barh(
    range(len(niches)), counts,
    color=colors, edgecolor='black', linewidth=0.5, height=0.6
)

# Add count labels
for i, (bar, count) in enumerate(zip(bars, counts)):
    ax1.text(
        count + 1, i, str(count),
        va='center', fontsize=11, fontweight='bold'
    )

ax1.set_yticks(range(len(niches)))
ax1.set_yticklabels(labels, fontsize=10)
ax1.set_xlabel('Number of unique ligand-receptor interactions', fontsize=11)
ax1.set_title('Unique interactions per niche\n(not shared with any other niche)', fontsize=11, fontweight='bold')
ax1.set_xlim(0, max(counts) * 1.15)
ax1.axvline(0, color='black', linewidth=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ── Panel B: top 5 unique interactions table ──────────────────────────
ax2 = fig.add_subplot(1, 2, 2)
ax2.axis('off')

y_start = 1.0
row_height = 0.038
header_height = 0.045

for niche in ['1', '2', '4', '5', '6']:
    df = top_unique[niche]
    color = niche_colors[niche]

    # Niche header
    ax2.text(
        0, y_start,
        niche_labels[niche].replace('\n', ' — '),
        fontsize=10, fontweight='bold', color='white',
        transform=ax2.transAxes,
        bbox=dict(boxstyle='round,pad=0.3', facecolor=color, edgecolor='none')
    )
    y_start -= header_height

    # Column headers
    ax2.text(0.0, y_start, 'Interaction', fontsize=8,
             fontweight='bold', transform=ax2.transAxes, color='grey')
    ax2.text(0.55, y_start, 'Target', fontsize=8,
             fontweight='bold', transform=ax2.transAxes, color='grey')
    ax2.text(0.75, y_start, 'Rank', fontsize=8,
             fontweight='bold', transform=ax2.transAxes, color='grey')
    y_start -= row_height * 0.8

    # Rows
    for _, row in df.iterrows():
        ax2.text(0.0, y_start, row['interaction'],
                 fontsize=8, transform=ax2.transAxes)
        ax2.text(0.55, y_start, row['target_label'],
                 fontsize=8, transform=ax2.transAxes)
        ax2.text(0.75, y_start, f"{row['magnitude_rank']:.4f}",
                 fontsize=8, transform=ax2.transAxes)
        y_start -= row_height

    y_start -= row_height * 0.5

ax2.set_title('Top 5 unique interactions per niche\n(ranked by magnitude)', fontsize=11, fontweight='bold')

plt.suptitle(
    'Niche-specific cell-cell communication landscape\nLIANA Rank Aggregate | consensus resource | magnitude_rank < 0.05',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('LIANA_unique_interactions_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved summary figure')

heatmap with niche labels:

In [ ]:
import liana as li
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Get results ───────────────────────────────────────────────────────
liana_res = adata_bio.uns['liana_res']
print(f"Total interactions: {len(liana_res)}")
print(f"Unique ligand-receptor pairs: {liana_res['ligand_complex'].nunique()}")

# Save full results
liana_res.to_csv('LIANA_results_all_niches.csv', index=False)
print('Saved full results')

# ── Filter significant interactions ───────────────────────────────────
# magnitude_rank < 0.05 is the standard significance threshold
liana_sig = liana_res[liana_res['magnitude_rank'] < 0.05].copy()
print(f"\nSignificant interactions (magnitude_rank < 0.05): {len(liana_sig)}")

# ── Heatmap: number of significant interactions per niche pair ─────────
liana_sig['pair'] = liana_sig['source'].astype(str) + '_' + liana_sig['target'].astype(str)
interaction_counts = liana_sig.groupby(['source', 'target']).size().reset_index(name='n_interactions')

# Pivot to matrix
pivot = interaction_counts.pivot(index='source', columns='target', values='n_interactions').fillna(0)

# Define the descriptive labels
niche_labels = {
    '1': 'Niche 1\nPeriurethral ductal',
    '2': 'Niche 2\nAggressive tumour',
    '4': 'Niche 4\nFibromuscular stroma',
    '5': 'Niche 5\nIncidental cancer',
    '6': 'Niche 6\nLuminal epithelium'
}

# Rename the rows (index) and columns of the pivot table
pivot = pivot.rename(index=niche_labels, columns=niche_labels)

# Increased figsize from (8, 6) to (10, 8) to fit larger labels
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    pivot,
    cmap='YlOrRd',
    annot=True, fmt='.0f',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Number of significant interactions'}
)
ax.set_title(
    'Cell-cell interactions per spatial niche pair\n(LIANA Rank Aggregate, magnitude_rank < 0.05)',
    fontsize=11, fontweight='bold', pad=15
)

# Set axes titles with some bolding for clarity
ax.set_xlabel('Target Niche', fontweight='bold', labelpad=10)
ax.set_ylabel('Source Niche', fontweight='bold', labelpad=10)

# Rotate x-ticks for readability so the long names don't overlap
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0) 

plt.tight_layout()
plt.savefig('LIANA_interaction_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved interaction heatmap')

# ── Dotplot: top interactions (Keeping your original code) ────────────
li.pl.dotplot(
    adata=adata_bio,
    colour='lr_means',
    size='specificity_rank',
    inverse_size=True,
    source_labels=['2', '5'],     # focus on tumour niches
    target_labels=['1', '4', '6'], # vs normal niches
    top_n=20,
    orderby='magnitude_rank',
    orderby_ascending=True,
    figure_size=(14, 8)
)
plt.savefig('LIANA_dotplot_tumour_vs_normal.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved dotplot')